In [ ]:
# librerias y config graficos

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def initialize_plotting():
    import matplotlib as mpl
    %config InlineBackend.figure_format = 'svg'
    label_size = 20
    mpl.rcParams['xtick.labelsize'] = label_size
    mpl.rcParams['ytick.labelsize'] = label_size
    mpl.rcParams['legend.fontsize'] = 14
    plt.rc('font', family='serif')
    mpl.rcParams.update({'font.size': 16})
    mpl.rcParams['text.usetex'] = False
    mpl.rcParams['figure.dpi'] = 120
initialize_plotting()

In [ ]:
# inciso (a) definimos todos los parámetrros y variables

g = 9.81 #gravedad
Omega = 7.2921e-5 # vel angular Tierra
m = 100 # masa en kg
Cd = 0.3 # coef de arrastre
A = 0.05 #area transversal m^2
rho = 1.225 #denidad aire 

#Condiciones iniciales
v1_0 = 500 #rapidez inicial
theta1_d = 45.0 #elevacion en grados
psi1_d = 30.0 #azimut en grados

tau = 10.0 #retardo segundos
r2_tau = np.array([5000.0, 2000.0, 0.0]) #pos inicial metros

phi_d = 45.0 #latitud del lanzamiento
d_tol = 10.0 #tolerancia

#juntamos todas las constantes de la fueza de roce
beta = (rho * Cd * A) / (2.0 * m)

#ponemos grados en radianes
theta1 = np.radians(theta1_d)
psi1 = np.radians(psi1_d)
phi = np.radians(phi_d)

In [ ]:
#inciso (b) simulamos numericamente la trayectoria
def dinamica_misil(t, Y):
    x, y, z, vx, vy, vz = Y

    v_mod = np.sqrt(vx**2 + vy**2 + vz**2) #modulo de v
    
    #derivadas de la posicion
    dxdt = vx
    dydt = vy
    dzdt = vz

    # aceleraciones
    dvx_dt = -beta * v_mod * vx - 2 * Omega * (vz * np.cos(phi) - vy * np.sin(phi))
    dvy_dt = -beta * v_mod * vy - 2 * Omega * (vx * np.sin(phi)) 
    dvz_dt = -g - beta * v_mod * vz + 2 * Omega * (vx * np.cos(phi))

    return [dxdt, dydt, dzdt, dvx_dt, dvy_dt, dvz_dt]

#detenemos la simulacion si el misil toca el suelo

def impacto_suelo(t, Y):
    return Y[2]

impacto_suelo.terminal = True #aborta cuando cruza el 0
impacto_suelo.direction = -1 # solo importa cuando va hacia abajo

# descomponemos el vector velocidad inicial usando trig. esférica
vx1_0 = v1_0 * np.cos(theta1) * np.cos(psi1)
vy1_0 = v1_0 * np.cos(theta1) * np.sin(psi1)
vz1_0 = v1_0 * np.sin(theta1)

# armamos el vector de estado inicial 
Y1_0 = [0.0, 0.0, 0.0, vx1_0, vy1_0, vz1_0]

# definimod el intervalo de tiempo para la simulación
t_span = (0, 150)
t_eval = np.linspace(t_span[0], t_span[1], 1000) # curva suave

# resolvemos la edo con RK45
solucion_misil_1 = solve_ivp(
    dinamica_misil, 
    t_span, 
    Y1_0, 
    method='RK45', 
    t_eval=t_eval, 
    events=impacto_suelo
)

# resultados RK
t_1 = solucion_misil_1.t
x_1 = solucion_misil_1.y[0]
y_1 = solucion_misil_1.y[1]
z_1 = solucion_misil_1.y[2]

print(f"Misil 1 impactó el suelo en t = {t_1[-1]:.2f} s")
print(f"Posición de impacto: x = {x_1[-1]:.2f} m, y = {y_1[-1]:.2f} m")

In [ ]:
# inciso (c)
from scipy.optimize import minimize
from scipy.interpolate import interp1d

# interpolamos trayectoria del misil 1
interp_x = interp1d(t_1, x_1, kind='cubic')
interp_y = interp1d(t_1, y_1, kind='cubic')
interp_z = interp1d(t_1, z_1, kind='cubic')

# parametros buscados
def error_intercepcion(params):
    v2, theta2, psi2, t_col = params
    
    # Penalizamos al algortimo si intenta probar tiempos absurdos
    
    if t_col <= tau or t_col > t_1[-1]:
        return 1e6 
        
    tiempo_vuelo_interceptor = t_col - tau
    
    # descomponemos velocidad 2
    vx2_0 = v2 * np.cos(theta2) * np.cos(psi2)
    vy2_0 = v2 * np.cos(theta2) * np.sin(psi2)
    vz2_0 = v2 * np.sin(theta2)
    
    Y2_0 = [r2_tau[0], r2_tau[1], r2_tau[2], vx2_0, vy2_0, vz2_0]
    
    # simulamos solo por el tiempo de vuelo del interceptor
    sol2 = solve_ivp(dinamica_misil, (0, tiempo_vuelo_interceptor), Y2_0, method='RK45')
    
    # posicion final interceptor
    pos_final_interceptor = np.array([sol2.y[0][-1], sol2.y[1][-1], sol2.y[2][-1]])
    
    # posicion final objetivo
    objetivo_xyz = np.array([interp_x(t_col), interp_y(t_col), interp_z(t_col)])
    
    # queremos que choquen
    return np.linalg.norm(pos_final_interceptor - objetivo_xyz)

# supongamos que queremos interceptarlo cerca de la mitad de su vuelo
t_guess = t_1[-1] / 2.0 
obj_guess = np.array([interp_x(t_guess), interp_y(t_guess), interp_z(t_guess)])

v2_guess = np.linalg.norm(obj_guess - r2_tau) / (t_guess - tau)
theta2_guess = np.arctan2(obj_guess[2] - r2_tau[2], np.linalg.norm(obj_guess[:2] - r2_tau[:2]))
psi2_guess = np.arctan2(obj_guess[1] - r2_tau[1], obj_guess[0] - r2_tau[0])

# le pasamos 4 variables al optimizador 
guess_inicial = [v2_guess, theta2_guess, psi2_guess, t_guess]


resultado_opt = minimize(error_intercepcion, guess_inicial, method='Nelder-Mead', tol=0.1)

# extraemos resultados
v2_opt, theta2_opt, psi2_opt, t_colision_opt = resultado_opt.x
distancia_minima = resultado_opt.fun

print(f"Distancia en el encuentro: {distancia_minima:.2f} m (Tolerancia: {d_tol} m)")
print(f"(c) Rapidez inicial v2_0: {v2_opt:.2f} m/s")
print(f"(c) Ángulo de elevación theta2: {np.degrees(theta2_opt):.2f}°")
print(f"(c) Azimut psi2: {np.degrees(psi2_opt):.2f}°")
print(f"(d) Tiempo de vuelo desde lanzamiento de M1: {t_colision_opt:.2f} s")

In [ ]:
#inciso (e) gráfico estático colisión

tiempo_vuelo_opt = t_colision_opt - tau #tiempo que el interc. estuvo en el aire

# descomponemos la vel 2
vx2_opt = v2_opt * np.cos(theta2_opt) * np.cos(psi2_opt)
vy2_opt = v2_opt * np.cos(theta2_opt) * np.sin(psi2_opt)
vz2_opt = v2_opt * np.sin(theta2_opt)

Y2_0_opt = [r2_tau[0], r2_tau[1], r2_tau[2], vx2_opt, vy2_opt, vz2_opt]
t_eval2 = np.linspace(0, tiempo_vuelo_opt, 500)

#resolvemos posicion de misil 2
sol2_opt  = solve_ivp(dinamica_misil, (0, tiempo_vuelo_opt), Y2_0_opt, method='RK45', t_eval=t_eval2)
x_2 = sol2_opt.y[0]
y_2 = sol2_opt.y[1]
z_2 = sol2_opt.y[2]

# donde esta el misil uno al momento de la colision
x_col = interp_x(t_colision_opt)
y_col = interp_y(t_colision_opt)
z_col = interp_z(t_colision_opt)

#grafico 3d
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

#buscamos el indice mas cercano al tiempo de colision
idx_colision = np.argmin(np.abs(t_1 - t_colision_opt))
ax.plot(x_1[:idx_colision], y_1[:idx_colision], z_1[:idx_colision], label='Misil 1 (Objetivo)', color='blue', lw=2)

ax.plot(x_2, y_2, z_2, label='Misil 2 (Interceptor)', color='orange', lw=2) #trayectoria misil 2
ax.scatter(0, 0, 0, color='blue', marker='o', s=50, label='Base M1')
ax.scatter(r2_tau[0], r2_tau[1], r2_tau[2], color='orange', marker='o', s=50, label='Base M2')
ax.scatter(x_col, y_col, z_col, color='red', marker='*', s=300, label='colisión')

ax.set_title(f'Intercepción Exitosa en t = {t_colision_opt:.2f} s', fontsize=14)
ax.set_xlabel('Este (x)')
ax.set_ylabel('Norte (y)')
ax.set_zlabel('Altura (z)')
ax.legend()
plt.show()

In [ ]:
#inciso (f) animación 3D

import matplotlib.animation as animation
from IPython.display import HTML

#config 
fig_anim = plt.figure(figsize=(10, 8))
ax_anim = fig_anim.add_subplot(111, projection='3d')
plt.rcParams['animation.embed_limit'] = 50.0
# limites fijos
ax_anim.set_xlim(0, 6000)
ax_anim.set_ylim(0, 5000)
ax_anim.set_zlim(0, 4000)
ax_anim.set_xlabel('Este (x)'); ax_anim.set_ylabel('Norte (y)'); ax_anim.set_zlabel('Altura (z)')

# elementos vacios que se iran actualizando
linea1, = ax_anim.plot([], [], [], 'b-', lw=2, label='M1 (Objetivo)')
punto1, = ax_anim.plot([], [], [], 'bo', ms=8)
linea2, = ax_anim.plot([], [], [], 'r-', lw=2, label='M2 (Interceptor)')
punto2, = ax_anim.plot([], [], [], 'ro', ms=8)
ax_anim.legend(loc='upper left')

# tiempos animacion
frames_totales = 150
t_anim = np.linspace(0, t_colision_opt, frames_totales)

#calculamos posiciones del misil 1
x1_anim = interp_x(t_anim)
y1_anim = interp_y(t_anim)
z1_anim = interp_z(t_anim)

#calculamos posiciones del misil 2
interp2_x = interp1d(sol2_opt.t + tau, x_2, kind='linear', fill_value=r2_tau[0], bounds_error=False)
interp2_y = interp1d(sol2_opt.t + tau, y_2, kind='linear', fill_value=r2_tau[1], bounds_error=False)
interp2_z = interp1d(sol2_opt.t + tau, z_2, kind='linear', fill_value=r2_tau[2], bounds_error=False)

#animación
def actualizar(i):
    # trayectoria misil 1
    linea1.set_data(x1_anim[:i], y1_anim[:i])
    linea1.set_3d_properties(z1_anim[:i])
    punto1.set_data([x1_anim[i]], [y1_anim[i]])
    punto1.set_3d_properties([z1_anim[i]])

    # trayectoria misil 2
    x2_i, y2_i, z2_i = interp2_x(t_anim[:i]), interp2_y(t_anim[:i]), interp2_z(t_anim[:i])
    linea2.set_data(x2_i, y2_i)
    linea2.set_3d_properties(z2_i)
    
    # misil 2 esperando en la base 
    if len(x2_i) > 0:
        punto2.set_data([x2_i[-1]], [y2_i[-1]])
        punto2.set_3d_properties([z2_i[-1]])
        
    return linea1, punto1, linea2, punto2

#animación resultante
anim = animation.FuncAnimation(fig_anim, actualizar, frames=frames_totales, interval=40, blit=False) 
plt.close(fig_anim)
HTML(anim.to_jshtml())
